## **Dog Re-Identification using DINOv2 Embeddings**

This notebook implements a prototype dog re-identification pipeline using pretrained DINOv2 embeddings and cosine similarity.

Goal:
Given a reference image of a dog, identify matching images from a query set.

This is treated as an image retrieval / re-identification problem rather than a breed classification task.

# 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision timm transformers scikit-learn matplotlib pillow tqdm umap-learn

# 2. Imports & Mount Drive

In [ ]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

from transformers import AutoImageProcessor, AutoModel

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.manifold import TSNE

from google.colab import drive
drive.mount('/content/drive')

dataset_dir = "/content/drive/MyDrive/dog_reid_dataset"

# 3. Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# 4. Load DINOv2

In [ ]:
model_name = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.to(device)
model.eval()

# 5. Image Preprocessing

In [ ]:
def load_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return image

# 6. Embedding Extraction

In [ ]:
@torch.no_grad()
def extract_embedding(image_path):
    image = load_image(image_path)

    inputs = processor(images=image, return_tensors="pt").to(device)

    outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0]

    embedding = F.normalize(embedding, p=2, dim=1)

    return embedding.cpu().numpy()[0]

# 7. Build Dataset Index

In [ ]:
embeddings = []
labels = []
paths = []

dog_folders = sorted(os.listdir(dataset_dir))

for dog_id in tqdm(dog_folders):

    folder_path = os.path.join(dataset_dir, dog_id)

    image_files = glob.glob(os.path.join(folder_path, "*.jpg"))

    for img_path in image_files:

        emb = extract_embedding(img_path)

        embeddings.append(emb)
        labels.append(dog_id)
        paths.append(img_path)

embeddings = np.array(embeddings)

print("Total embeddings:", len(embeddings))

# 8. Similarity Search Function

In [ ]:
def retrieve_similar(query_path, top_k=5):

    query_embedding = extract_embedding(query_path)

    sims = cosine_similarity(
        [query_embedding],
        embeddings
    )[0]

    ranked_indices = np.argsort(sims)[::-1]

    ranked_indices = [
        idx for idx in ranked_indices
        if paths[idx] != query_path
    ]

    results = []

    for idx in ranked_indices[:top_k]:

        results.append({
            "path": paths[idx],
            "label": labels[idx],
            "score": sims[idx]
        })

    return results

# 9. Visualize Retrieval Results

In [ ]:
def show_results(query_path, top_k=5):

    results = retrieve_similar(query_path, top_k)

    fig, axes = plt.subplots(1, top_k + 1, figsize=(15, 5))

    query_img = load_image(query_path)

    axes[0].imshow(query_img)
    axes[0].set_title("Query")
    axes[0].axis("off")

    for i, result in enumerate(results):

        img = load_image(result["path"])

        axes[i + 1].imshow(img)

        axes[i + 1].set_title(
            f"{result['label']}\n{result['score']:.3f}"
        )

        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()

# 10. Test Example

In [ ]:
query_path = paths[0]

show_results(query_path)

# 11. Evaluation Setup

In [ ]:
pairs = []
y_true = []
scores = []

for i in tqdm(range(len(paths))):

    for j in range(i + 1, len(paths)):

        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[j]]
        )[0][0]

        scores.append(sim)

        same = int(labels[i] == labels[j])

        y_true.append(same)

# 12. Threshold Classification

In [ ]:
threshold = 0.75

y_pred = [1 if s >= threshold else 0 for s in scores]

# 13. Metrics

In [ ]:
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, scores))

# 14. Rank-1 Accuracy

In [ ]:
correct = 0
total = 0

for idx, query_path in enumerate(paths):

    query_label = labels[idx]

    results = retrieve_similar(query_path, top_k=2)

    top_match = results[1]

    if top_match["label"] == query_label:
        correct += 1

    total += 1

rank1 = correct / total

print("Rank-1 Accuracy:", rank1)

# 15. Similarity Distribution Plot

In [ ]:
positive_scores = [
    s for s, y in zip(scores, y_true) if y == 1
]

negative_scores = [
    s for s, y in zip(scores, y_true) if y == 0
]

plt.figure(figsize=(8,5))

plt.hist(positive_scores, bins=30, alpha=0.6, label="Same Dog")
plt.hist(negative_scores, bins=30, alpha=0.6, label="Different Dog")

plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.legend()

plt.title("Similarity Score Distribution")

plt.show()

# 16. Embedding Visualization

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=10,
    random_state=42
)

reduced = tsne.fit_transform(embeddings)

In [ ]:
plt.figure(figsize=(10,8))

unique_labels = list(set(labels))

for label in unique_labels:

    idxs = [i for i, l in enumerate(labels) if l == label]

    plt.scatter(
        reduced[idxs, 0],
        reduced[idxs, 1],
        label=label
    )

plt.legend(bbox_to_anchor=(1.05, 1))
plt.title("t-SNE of Dog Embeddings")

plt.show()

# 17. Find Failure Cases

In [ ]:
failure_cases = []

for idx, query_path in enumerate(paths):

    query_label = labels[idx]

    results = retrieve_similar(query_path, top_k=1)

    top_match = results[0]

    predicted_label = top_match["label"]

    if predicted_label != query_label:

        failure_cases.append({
            "query_path": query_path,
            "query_label": query_label,
            "match_path": top_match["path"],
            "match_label": predicted_label,
            "score": top_match["score"]
        })

print("Total failure cases:", len(failure_cases))

In [ ]:
def show_failure_case(case):

    query_img = load_image(case["query_path"])
    match_img = load_image(case["match_path"])

    fig, axes = plt.subplots(1, 2, figsize=(10,5))

    axes[0].imshow(query_img)
    axes[0].set_title(
        f"Query\n{case['query_label']}"
    )
    axes[0].axis("off")

    axes[1].imshow(match_img)
    axes[1].set_title(
        f"Incorrect Match\n{case['match_label']}\nScore: {case['score']:.3f}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
for case in failure_cases[:5]:
    show_failure_case(case)

## Failure Analysis

Observed failure modes:
1. Similar coat patterns between different dogs
2. Extreme pose variation
3. Occlusions and low-resolution images
4. Lighting inconsistencies